<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install deepxde
!pip install optuna

In [ ]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import optuna
import plotly.express as px
import gc
import time

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [ ]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [ ]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [ ]:
def exact_solution(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

In [ ]:
iterations = 10000
epsilon = 1e-4

In [ ]:
def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [ ]:
XT_test = geomtime.uniform_points(2500)
y_true = exact_solution(XT_test)

In [ ]:
fig = px.scatter(
    x=XT_test[:, 0],
    y=XT_test[:, 1],
    labels={"x": "Координата x", "y": "Время t"}
)

fig.add_shape(
    type="rect",
    x0=XT_test[:, 0].min(), y0=XT_test[:, 1].min(),
    x1=XT_test[:, 0].max(), y1=XT_test[:, 1].max(),
    line=dict(color="blue", width=2),
    fillcolor="rgba(0, 0, 255, 0.05)",
)

fig.update_layout(template="plotly_white")
fig.update_traces(marker=dict(size=6, color="red"))
fig.show()

In [ ]:
def objective(trial):
  start_time = time.time()
  dde.config.set_random_seed(42)
  num_domain = trial.suggest_int("num_domain", 200, 5000)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=num_domain, train_distribution="uniform")

  net = dde.nn.FNN([2] + [20] * 3 + [1], "tanh", "Glorot normal")
  net.apply_output_transform(output_transform)

  model = dde.Model(data, net)
  model.compile("adam", lr=1e-3)
  model.train(iterations=iterations, display_every=500)

  model.compile("L-BFGS")
  model.train()

  y_pred = model.predict(XT_test)
  l2_error = dde.metrics.l2_relative_error(y_true, y_pred)

  end_time = time.time() - start_time

  del model, net, data
  gc.collect()
  torch.cuda.empty_cache()
  return float(l2_error), float(end_time)

study = optuna.create_study(directions=["minimize", "minimize"])
study.optimize(objective, n_trials=5)

valid_trials = [t for t in study.best_trials if t.values[0] < epsilon]
valid_trials_sorted = sorted(valid_trials, key=lambda t: t.values[1])

for t in valid_trials_sorted[:3]:
  print(t.params, t.values)

[I 2026-11-16 07:23:21,086] A new study created in memory with name: no-name-d0b9f33f-5d4d-4066-9ade-6ae2f25cffbe


Compiling model...
'compile' took 5.621370 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.55e+00]    [1.55e+00]    []  
500       [1.08e-03]    [1.08e-03]    []  
1000      [3.32e-04]    [3.32e-04]    []  
1500      [1.24e-04]    [1.24e-04]    []  
2000      [5.89e-05]    [5.89e-05]    []  
2500      [3.41e-05]    [3.41e-05]    []  
3000      [2.30e-05]    [2.30e-05]    []  
3500      [1.70e-05]    [1.70e-05]    []  
4000      [1.32e-05]    [1.32e-05]    []  
4500      [1.04e-05]    [1.04e-05]    []  
5000      [8.52e-06]    [8.52e-06]    []  
5500      [7.30e-06]    [7.30e-06]    []  
6000      [6.31e-06]    [6.31e-06]    []  
6500      [1.73e-05]    [1.73e-05]    []  
7000      [8.01e-05]    [8.01e-05]    []  
7500      [6.12e-06]    [6.12e-06]    []  
8000      [4.39e-06]    [4.39e-06]    []  
8500      [4.51e-06]    [4.51e-06]    []  
9000      [3.80e-06]    [3.80e-06]    []  
9500      [3.55e-06]    [3.55e-06]    []  
10000     [3.26e-06]    [

[I 2026-11-16 07:25:19,166] Trial 0 finished with values: [0.00014637247659265995, 117.73135495185852] and parameters: {'num_domain': 2580}.


Compiling model...
'compile' took 0.000156 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.54e+00]    [1.54e+00]    []  
500       [1.07e-03]    [1.07e-03]    []  
1000      [3.30e-04]    [3.30e-04]    []  
1500      [1.23e-04]    [1.23e-04]    []  
2000      [5.85e-05]    [5.85e-05]    []  
2500      [3.39e-05]    [3.39e-05]    []  
3000      [2.29e-05]    [2.29e-05]    []  
3500      [1.70e-05]    [1.70e-05]    []  
4000      [1.32e-05]    [1.32e-05]    []  
4500      [1.04e-05]    [1.04e-05]    []  
5000      [8.45e-06]    [8.45e-06]    []  
5500      [7.31e-06]    [7.31e-06]    []  
6000      [6.30e-06]    [6.30e-06]    []  
6500      [5.68e-06]    [5.68e-06]    []  
7000      [5.21e-06]    [5.21e-06]    []  
7500      [2.82e-05]    [2.82e-05]    []  
8000      [4.49e-06]    [4.49e-06]    []  
8500      [4.08e-06]    [4.08e-06]    []  
9000      [3.81e-06]    [3.81e-06]    []  
9500      [3.56e-06]    [3.56e-06]    []  
10000     [3.23e-06]    [

[I 2026-11-16 07:27:09,130] Trial 1 finished with values: [9.778579260455444e-05, 109.50546741485596] and parameters: {'num_domain': 3588}.


Compiling model...
'compile' took 0.000252 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.55e+00]    [1.55e+00]    []  
500       [1.08e-03]    [1.08e-03]    []  
1000      [3.34e-04]    [3.34e-04]    []  
1500      [1.25e-04]    [1.25e-04]    []  
2000      [5.93e-05]    [5.93e-05]    []  
2500      [3.43e-05]    [3.43e-05]    []  
3000      [2.31e-05]    [2.31e-05]    []  
3500      [1.71e-05]    [1.71e-05]    []  
4000      [1.32e-05]    [1.32e-05]    []  
4500      [1.04e-05]    [1.04e-05]    []  
5000      [8.43e-06]    [8.43e-06]    []  
5500      [7.20e-06]    [7.20e-06]    []  
6000      [7.61e-05]    [7.61e-05]    []  
6500      [2.13e-05]    [2.13e-05]    []  
7000      [5.21e-06]    [5.21e-06]    []  
7500      [1.95e-04]    [1.95e-04]    []  
8000      [4.41e-06]    [4.41e-06]    []  
8500      [6.53e-04]    [6.53e-04]    []  
9000      [3.78e-06]    [3.78e-06]    []  
9500      [3.52e-06]    [3.52e-06]    []  
10000     [3.21e-06]    [

... (2 outputs left out)

[I 2026-11-16 07:30:44,275] Trial 3 finished with values: [0.00028236201615072787, 104.43770098686218] and parameters: {'num_domain': 3131}.


Compiling model...
'compile' took 0.000149 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.69e+00]    [1.69e+00]    []  
500       [1.18e-03]    [1.18e-03]    []  
1000      [3.77e-04]    [3.77e-04]    []  
1500      [1.41e-04]    [1.41e-04]    []  
2000      [6.62e-05]    [6.62e-05]    []  
2500      [3.80e-05]    [3.80e-05]    []  
3000      [2.50e-05]    [2.50e-05]    []  
3500      [1.79e-05]    [1.79e-05]    []  
4000      [1.35e-05]    [1.35e-05]    []  
4500      [1.04e-05]    [1.04e-05]    []  
5000      [8.22e-06]    [8.22e-06]    []  
5500      [6.85e-06]    [6.85e-06]    []  
6000      [6.00e-06]    [6.00e-06]    []  
6500      [5.24e-06]    [5.24e-06]    []  
7000      [1.24e-03]    [1.24e-03]    []  
7500      [4.38e-06]    [4.38e-06]    []  
8000      [4.09e-06]    [4.09e-06]    []  
8500      [7.89e-04]    [7.89e-04]    []  
9000      [3.53e-06]    [3.53e-06]    []  
9500      [5.55e-06]    [5.55e-06]    []  
10000     [3.15e-06]    [

[I 2026-11-16 07:32:28,517] Trial 4 finished with values: [0.0005962871364317834, 103.89664435386658] and parameters: {'num_domain': 247}.


{'num_domain': 3588} [9.778579260455444e-05, 109.50546741485596]
